In [27]:
import pickle

x_train=pickle.load(open("x_train.pk1","rb"))
x_test=pickle.load(open("x_test.pk1","rb"))
y_train=pickle.load(open("y_train.pk1","rb"))
y_test=pickle.load(open("y_test.pk1","rb"))

In [28]:
from sklearn.tree import DecisionTreeRegressor
regressor=DecisionTreeRegressor()
from sklearn.metrics import r2_score

experiments = pd.read_excel("DT model Hypertuning - R2 score.xlsx")
for index, row in experiments.iterrows():
    criterion = row["Criterion"]
    splitter = row["Splitter"]
    max_features = row["Max Features"]
    max_depth = row["Max Depth"]
    min_split = int(row["Min Split"])
    min_leaf = int(row["Min Leaf"])
    ccp_alpha = float(row["CCP Alpha"])
    
    if pd.isna(max_depth) or str(max_depth).lower() == "none":
        max_depth = None
    if pd.isna(max_features) or str(max_features).lower() == "none":
        max_features = None
    
    regressor = DecisionTreeRegressor(
    criterion=criterion,
    splitter=splitter,
    max_features=max_features,
    max_depth=max_depth,
    min_samples_split=min_split,
    min_samples_leaf=min_leaf,
    ccp_alpha=ccp_alpha,
    random_state=0)
    
    #Train
    import time
    start = time.time()
    regressor.fit(x_train, y_train)
    end = time.time()
    train_time = (end-start)*1000
    
    #Predict
    train_pred = regressor.predict(x_train)
    test_pred = regressor.predict(x_test)
    
    #Metrics
    train_r2 = r2_score(y_train, train_pred)
    test_r2 = r2_score(y_test, test_pred)
    from sklearn.metrics import mean_absolute_error
    from sklearn.metrics import mean_squared_error
    mae = mean_absolute_error(y_test, test_pred)
    rmse = mean_squared_error(y_test,test_pred,squared=False)
    
    # Prediction Time
    start = time.time()
    regressor.predict(x_test)
    end = time.time()
    prediction_time = (end-start)*1000

    # Tree Details
    depth = regressor.get_depth()
    leaf_nodes = regressor.get_n_leaves()

    # Overfitting
    difference = train_r2 - test_r2

    if difference < 0.03:
        status = "Low"
    elif difference < 0.10:
        status = "Moderate"
    else:
        status = "High"

    # Write values back to DataFrame
    experiments.loc[index, "Train R²"] = round(train_r2,4)
    experiments.loc[index, "Test R²"] = round(test_r2,4)
    experiments.loc[index, "MAE"] = round(mae,2)
    experiments.loc[index, "RMSE"] = round(rmse,2)
    experiments.loc[index, "Train Time(ms)"] = round(train_time,2)
    experiments.loc[index, "Predict Time(ms)"] = round(prediction_time,2)
    experiments.loc[index, "Tree Depth"] = depth
    experiments.loc[index, "Leaf Nodes"] = leaf_nodes
    experiments.loc[index, "Overfitting"] = status



In [29]:
#Ranking
experiments = experiments.sort_values(by="Test R²",ascending=False)
experiments["Rank"] = range(1,len(experiments)+1)


#Saving to excel
experiments.to_excel("DecisionTreeResults.xlsx",index=False)
print("Excel saved successfully!")



Excel saved successfully!


In [30]:
import pandas as pd

df = pd.read_excel("DecisionTreeResults.xlsx")

print(df.head())

   SL.NO       Criterion Max Features Splitter Max Depth  Min Split  Min Leaf  \
0      7  absolute_error         None     best      None          2         1   
1      9   squared_error         None     best         5          2         1   
2      1   squared_error         None     best      None          2         1   
3     10   squared_error         None     best         8          2         1   
4     11   squared_error         None     best         5          5         2   

   CCP Alpha  Train R²  Test R²      MAE     RMSE  Train Time(ms)  \
0        0.0    1.0000   0.9330  5746.77  8044.15            0.00   
1        0.0    0.9991   0.9220  7006.74  8675.94            8.21   
2        0.0    1.0000   0.9087  7474.49  9386.00            8.06   
3        0.0    1.0000   0.9087  7474.49  9386.00           15.62   
4        0.0    0.9850   0.9000  8030.90  9823.87            0.00   

   Predict Time(ms)  Tree Depth  Leaf Nodes Overfitting  Rank  
0              0.00           8   